# Urdu OCR Project - Week 3
## Expand Dataset + Build Data Loader

## Step 1: Verify Dataset Size

In [6]:
import pandas as pd
import os

df = pd.read_csv('data/labels.csv')
print(f'Total images in labels.csv: {len(df)}')
print(df.head())

missing = [p for p in df['image'] if not os.path.exists(p)]
print(f'Missing files: {len(missing)}')


Total images in labels.csv: 220
                   image                        text
0  data/raw/img_0000.png  میں اردو زبان سیکھ رہا ہوں
1  data/raw/img_0001.png  میں اردو زبان سیکھ رہا ہوں
2  data/raw/img_0002.png              یہ ایک مثال ہے
3  data/raw/img_0003.png  پاکستان ایک خوبصورت ملک ہے
4  data/raw/img_0004.png  میں اردو زبان سیکھ رہا ہوں
Missing files: 0


## Step 2: Install Dependencies

In [7]:
!pip install transformers torch pillow pandas

## Step 3: Build the Dataset Class

In [8]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import TrOCRProcessor
from PIL import Image
import pandas as pd


class UrduOCRDataset(Dataset):
    def __init__(self, csv_path, processor):
        self.data = pd.read_csv(csv_path)
        self.processor = processor
        print(f'Dataset loaded: {len(self.data)} samples')

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        image = Image.open(row['image']).convert('RGB')

        encoding = self.processor(image, return_tensors='pt')
        pixel_values = encoding.pixel_values.squeeze()

        labels = self.processor.tokenizer(
            row['text'],
            padding='max_length',
            max_length=128
        ).input_ids

        labels = torch.tensor(labels)

        return {'pixel_values': pixel_values, 'labels': labels}


## Step 4: Load Processor and Test Dataset

In [9]:
processor = TrOCRProcessor.from_pretrained('microsoft/trocr-base-printed')

dataset = UrduOCRDataset('data/labels.csv', processor)

sample = dataset[0]
print('Sample pixel_values shape:', sample['pixel_values'].shape)
print('Sample labels shape:', sample['labels'].shape)
print('Dataset is working correctly!')


preprocessor_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.13k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

Dataset loaded: 220 samples
Sample pixel_values shape: torch.Size([3, 384, 384])
Sample labels shape: torch.Size([128])
Dataset is working correctly!


## Step 5: Train/Test Split

In [10]:
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = torch.utils.data.random_split(
    dataset, [train_size, test_size]
)

print(f'Training samples: {train_size}')
print(f'Testing samples: {test_size}')


Training samples: 176
Testing samples: 44


## Step 6: Build DataLoader

In [11]:
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

batch = next(iter(train_loader))
print('Batch pixel_values shape:', batch['pixel_values'].shape)
print('Batch labels shape:', batch['labels'].shape)
print('DataLoader is working correctly!')


Batch pixel_values shape: torch.Size([4, 3, 384, 384])
Batch labels shape: torch.Size([4, 128])
DataLoader is working correctly!


## Summary

In [12]:
print(f'Total dataset size: {len(dataset)}')
print(f'Training samples: {train_size}')
print(f'Testing samples: {test_size}')
print('My dataset has', len(dataset), 'images and loads correctly')


Total dataset size: 220
Training samples: 176
Testing samples: 44
My dataset has 220 images and loads correctly
